### Step 1: Data Ingestion and Pre-processing
<p>This section contains the data loader with support for doc, docx and pdf file formats.<br>
<b>Step 1: Data Loading and Serialization</b><br>
All files contained within the INPUT_FOLDER specified at the top are loaded, parsed, serialized to JSON and saved to the <u>parsed</u> folder.<br>
<b>Step 2: Data Cleaning</b><br>
All JSON files within the parsed folders are cleaned in-place to remove unncessary whitespaces and elements with empty text.<br>
<b>Step 3: Data Normalization</b><br>
All cleaned JSON files within the parsed folders are now normalized to have a uniform 4 key schema, which includes type, text, coursename and coursecode and stored in the <u>normalized</u> folder.
</p>

In [ ]:
import os

# --- import your functions ---
from src.ingestion_and_preprocessing import *

# --- folder containing raw documents ---
INPUT_FOLDER = "dataset"

# --- process all files ---
for file_name in os.listdir(INPUT_FOLDER):
    file_path = os.path.join(INPUT_FOLDER, file_name)

    # skip non-files
    if not os.path.isfile(file_path):
        continue

    # --- only supported formats ---
    if not file_name.lower().endswith((".pdf", ".doc", ".docx")):
        continue

    print(f"\nProcessing: {file_name}")

    try:
        # 1. Partition → saves to parsed/
        partitioner(file_path)

        # --- construct parsed file path ---
        name, ext = os.path.splitext(file_name)
        ext = ext.lower()

        if ext == ".pdf":
            parsed_path = f"parsed/{name}_pdf.json"
        elif ext == ".doc":
            parsed_path = f"parsed/{name}_doc.json"
        elif ext == ".docx":
            parsed_path = f"parsed/{name}_docx.json"

        # 2a. Clean (in-place)
        clean_json_elements(parsed_path)

        # 2b. Table → Markdown (in-place)
        convert_tables_to_markdown(parsed_path)

        # 3. Normalize → saves to normalized/
        normalize_json_elements(parsed_path)

    except Exception as e:
        print(f"Error processing {file_name}: {e}")

### Step 2: Chunking
<p> Fixed token chunking strategy was employed with 2 chunk sizes and overlaps: <b>256/26</b> and <b>512/52</b>.<br>
Chunking was done by employing two types of tokenizers as per those used by our embedding models, namely <i>sentence-transformers/all-MiniLM-L6-v2</i> and <i>BAAI/bge-base-en-v1.5</i>.<br>
Three types of chunks were made: 256/26 chunk for both MiniLM-L6-v2 and bge-base-en-v1.5 and then 512/52 for bge-base-en-v1.5 alone.
</p>

In [ ]:
from src.chunking import chunking_minilm_l6_v2

chunking_minilm_l6_v2(
    input_folder="normalized",
    chunk_size=256,
    overlap=26
)

In [ ]:
from src.chunking import chunking_bge_base_v1_5

chunking_bge_base_v1_5(
    input_folder="normalized",
    chunk_size=256,
    overlap=26
)

In [ ]:
chunking_bge_base_v1_5(
    input_folder="normalized",
    chunk_size=512,
    overlap=52
)

### Step 3: Embedding and Vector Database Storage

In [ ]:
from src.embedding_qdrant import embed_and_store

embed_and_store(
    chunk_folder="chunks/minilm_l6_v2_256_26",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    collection_name="minilm_l6_v2_256_26",
)

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_256_26",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_256_26",
)

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_512_52",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

In [ ]:
from src.embedding_chromadb import embed_and_store

embed_and_store(
    chunk_folder="chunks/bge_base_v1_5_512_52",
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

### Step 4: LLM Prompting and Response Generation

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_qdrant(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="llama3.1:8b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="llama3.1:8b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="gemma2:9b",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

In [ ]:
from src.rag_prompt import rag_inference
from src.query import *

query = "What is the assessment pattern in detail with all components of generative ai?"

chunks = query_chromadb(
    query=query,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

answer = rag_inference(
    model_name="phi3:mini",
    retrieved_chunks=chunks,
    query=query
)

print(answer)

### Retrieval Quality Evaluation

In [ ]:
# from src.retrieval_evaluation import *

# generate_eval_dataset(
#     normalized_folder="normalized",
#     num_files=15,
#     samples_per_file=3,
#     model_name="mistral:7b-instruct ",
#     save_path="eval_dataset.json"
# )

[1/15] Processing file: Discrete Mathematics_CSE3715.json...
[2/15] Processing file: Computer Organization & Architecture_CSE2709.json...
[3/15] Processing file: Project-III_PRJ3902.json...
[4/15] Processing file: Seminar Case Studies_SEM3501.json...
[5/15] Processing file: Software Engineering_CSE3005.json...
[6/15] Processing file: Digital Image Processing_CSE3022.json...
[7/15] Processing file: Logical Reasoning & Quantitative Analysis_PSP4706.json...
[8/15] Processing file: Generative AI and LLMs_CSE3720.json...
[9/15] Processing file: Innovation and Entrepreneurship_ENT3005.json...
[10/15] Processing file: Cryptography_CSE3703.json...
[11/15] Processing file: Financial Markets From Fundamentals to AI-Driven Analytics_FIN1001.json...
[12/15] Processing file: Making Causal Inferences to Support Decisions_CSE3025.json...
[13/15] Processing file: Practice School-II_PSI2501.json...
[14/15] Processing file: Network Security_CSE3714.json...
[15/15] Processing file: Global Energy Politics

[{'question': 'What is being discussed in this text?',
  'ground_truth_answer': 'Understanding.',
  'reference_text': '• Understanding of',
  'source_file': 'Discrete Mathematics_CSE3715'},
 {'question': "What is the URL for the Discrete Mathematics course taught at Oxford University's Computer Science department?",
  'ground_truth_answer': '<https://www.cs.ox.ac.uk/teaching/courses/discretemaths/>',
  'reference_text': '1. https://www.cs.ox.ac.uk/teaching/courses/discretemaths/',
  'source_file': 'Discrete Mathematics_CSE3715'},
 {'question': 'What are the topics discussed in the text?',
  'ground_truth_answer': 'The text discusses topics related to Permutations and Combinations, as well as Combinatorial Arguments.',
  'reference_text': '- Permutations and Combinations, Combinatorial Arguments',
  'source_file': 'Discrete Mathematics_CSE3715'},
 {'question': 'What is the title of the book by William Stallings published in 2013?',
  'ground_truth_answer': '"Computer Organization and Ar

refined manually

<b>Comparing models in same db, same chunk size and overlap</b>

In [9]:
from src.retrieval_evaluation import *
from src.query import *

evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    collection_name="minilm_l6_v2_256_26"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6272.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4997.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8434.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embedd


📊 Retrieval Evaluation Results:
total_samples: 35
hit_rate@k: 0.8286
mrr: 0.8286
recall@k: 0.8286


{'total_samples': 35,
 'hit_rate@k': 0.8285714285714286,
 'mrr': 0.8285714285714286,
 'recall@k': 0.8285714285714286}

In [4]:
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_256_26",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7167.78it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5705.95it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5328.57it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
hit_rate@k: 0.8857
mrr: 0.8857
recall@k: 0.8857


{'total_samples': 35,
 'hit_rate@k': 0.8857142857142857,
 'mrr': 0.8857142857142857,
 'recall@k': 0.8857142857142857}

<b>Comparing chunk sizes w same model, same db</b>

In [5]:
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5716.27it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5734.06it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4490.40it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
hit_rate@k: 0.9143
mrr: 0.9143
recall@k: 0.9143


{'total_samples': 35,
 'hit_rate@k': 0.9142857142857143,
 'mrr': 0.9142857142857143,
 'recall@k': 0.9142857142857143}

<b>Comparing vector dbs w same model, same chunk size</b>

In [6]:
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_chromadb,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6670.45it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6331.67it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6463.92it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
hit_rate@k: 0.9143
mrr: 0.9143
recall@k: 0.9143


{'total_samples': 35,
 'hit_rate@k': 0.9142857142857143,
 'mrr': 0.9142857142857143,
 'recall@k': 0.9142857142857143}

<p>1. MiniLM 256 Qdrant:<br>
'total_samples': 35, 'hit_rate@k': 0.8285714285714286
</p>
<br>
<p>2. BGE 256 Qdrant:<br>
'total_samples': 35, 'hit_rate@k': 0.8857142857142857</p>
<br>
<p>3. BGE 512 Qdrant:<br>
'total_samples': 35, 'hit_rate@k': 0.9142857142857143
</p>
<br>
<p>4. BGE 512 ChromaDB:<br>
total_samples: 35
hit_rate@k: 0.9143
</p>

## NEW EVAL

<p>1. MiniLM 256 Qdrant:<br>
{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.8285714285714286,
 'mrr': 0.6833333333333332,
 'answer_coverage@k': 0.8285714285714286}
</p>
<br>
<p>2. BGE 256 Qdrant:<br>
{'total_samples': 35,
 'top1_accuracy': 0.6571428571428571,
 'hit_rate@k': 0.8857142857142857,
 'mrr': 0.7595238095238095,
 'answer_coverage@k': 0.8857142857142857} </p>
<br>
<p>3. BGE 512 Qdrant:<br>
{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.9428571428571428,
 'mrr': 0.7309523809523809,
 'answer_coverage@k': 0.9142857142857143}
</p>
<br>
<p>4. BGE 512 ChromaDB:<br>
{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.9428571428571428,
 'mrr': 0.7309523809523809,
 'answer_coverage@k': 0.9142857142857143}
</p>

In [5]:
from src.retrieval_evaluation import *
from src.query import *
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="sentence-transformers/all-MiniLM-L6-v2", # Use full name
    collection_name="minilm_l6_v2_256_26")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5691.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6511.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5342.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embedd


📊 Retrieval Evaluation Results:
total_samples: 35
top1_accuracy: 0.5714
hit_rate@k: 0.8286
mrr: 0.6833
answer_coverage@k: 0.8286


{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.8285714285714286,
 'mrr': 0.6833333333333332,
 'answer_coverage@k': 0.8285714285714286}

In [6]:
from src.retrieval_evaluation import *
from src.query import *
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_256_26",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5705.37it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5356.16it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5239.19it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
top1_accuracy: 0.6571
hit_rate@k: 0.8857
mrr: 0.7595
answer_coverage@k: 0.8857


{'total_samples': 35,
 'top1_accuracy': 0.6571428571428571,
 'hit_rate@k': 0.8857142857142857,
 'mrr': 0.7595238095238095,
 'answer_coverage@k': 0.8857142857142857}

In [7]:
from src.retrieval_evaluation import *
from src.query import *
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_qdrant,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6066.24it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6968.97it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5954.08it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
top1_accuracy: 0.5714
hit_rate@k: 0.9429
mrr: 0.7310
answer_coverage@k: 0.9143


{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.9428571428571428,
 'mrr': 0.7309523809523809,
 'answer_coverage@k': 0.9142857142857143}

In [8]:
from src.retrieval_evaluation import *
from src.query import *
evaluate_retrieval(
    eval_dataset_path="eval_dataset.json",
    retrieval_fn=query_chromadb,
    model_name="BAAI/bge-base-en-v1.5",
    collection_name="bge_base_v1_5_512_52",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5279.16it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5978.30it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5838.50it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEX


📊 Retrieval Evaluation Results:
total_samples: 35
top1_accuracy: 0.5714
hit_rate@k: 0.9429
mrr: 0.7310
answer_coverage@k: 0.9143


{'total_samples': 35,
 'top1_accuracy': 0.5714285714285714,
 'hit_rate@k': 0.9428571428571428,
 'mrr': 0.7309523809523809,
 'answer_coverage@k': 0.9142857142857143}